# Chapter 10 - Evaluation Metrics and Model Tuning

Accuracy alone is a dangerously incomplete picture of classifier performance. A model
that predicts "no fraud" for every transaction achieves 99.9% accuracy on a dataset
where only 0.1% of transactions are fraudulent — and catches zero fraud.

This notebook dives deep into the metrics that actually matter, how to visualize
tradeoffs, and how to tune logistic regression for real-world performance.

**Topics covered:**
- Confusion matrix: TP, TN, FP, FN
- Precision, Recall, F1-score
- ROC curve and AUC
- Precision-Recall curve
- Threshold tuning for business metrics
- Handling class imbalance
- Regularization: C parameter, L1 vs L2
- Hyperparameter tuning with GridSearchCV

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    classification_report,
)

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

In [ ]:
# Generate a slightly imbalanced dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.7, 0.3],
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples, class distribution: {np.bincount(y_train)}')
print(f'Test:  {X_test.shape[0]} samples, class distribution: {np.bincount(y_test)}')

In [ ]:
# Train baseline model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Baseline accuracy: {accuracy_score(y_test, y_pred):.4f}')

## The Confusion Matrix

The confusion matrix is the foundation of all classification metrics. It counts the
four possible outcomes for each prediction:

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | True Positive (TP) | False Negative (FN) |
| **Actually Negative** | False Positive (FP) | True Negative (TN) |

- **TP:** Correctly predicted positive (hit)
- **TN:** Correctly predicted negative (correct rejection)
- **FP:** Incorrectly predicted positive (false alarm, Type I error)
- **FN:** Missed a positive (miss, Type II error)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'True Positives:  {tp}')
print(f'True Negatives:  {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

### Deriving Metrics from the Confusion Matrix

All the key metrics are just different ratios of TP, TN, FP, and FN:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of all predicted positives, how many are correct?)}$$

$$\text{Recall (Sensitivity)} = \frac{TP}{TP + FN} \quad \text{(of all actual positives, how many did we catch?)}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(harmonic mean of precision and recall)}$$

In [ ]:
# Compute metrics manually from confusion matrix values
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print('Manual computation from confusion matrix:')
print(f'  Accuracy:  {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1 Score:  {f1:.4f}')
print()
print('Verification with sklearn:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred):.4f}')
print(f'  Recall:    {recall_score(y_test, y_pred):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred):.4f}')

## Precision vs Recall Tradeoff

Precision and recall are in **tension** with each other. You cannot improve one without
(usually) hurting the other.

- **High precision, low recall:** The model is conservative — it only predicts positive
  when very confident, so it misses many true positives.
- **High recall, low precision:** The model is aggressive — it catches most positives
  but also flags many negatives as positive.

The right balance depends on the **cost of each error type** in your specific application.

In [ ]:
# Demonstrate the tradeoff by varying the threshold
thresholds = np.arange(0.1, 0.95, 0.05)
precisions = []
recalls = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    # Avoid division by zero: if no positive predictions, precision is undefined
    if y_pred_t.sum() == 0:
        precisions.append(1.0)  # convention
    else:
        precisions.append(precision_score(y_test, y_pred_t, zero_division=1))
    recalls.append(recall_score(y_test, y_pred_t))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, 'b-o', markersize=4, label='Precision')
ax.plot(thresholds, recalls, 'r-o', markersize=4, label='Recall')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall at Different Thresholds')
ax.legend()
plt.tight_layout()
plt.show()

| Application | Priority | Why |
|---|---|---|
| Cancer screening | **High recall** | Missing a cancer case (FN) is far worse than a false alarm (FP) |
| Spam filter | **High precision** | Sending real email to spam (FP) is very annoying |
| Fraud detection | **Balance** | Missing fraud is costly, but too many false alerts overwhelm investigators |
| Legal discovery | **High recall** | Must find all relevant documents, even at cost of reviewing extras |

## The ROC Curve and AUC

The **Receiver Operating Characteristic (ROC) curve** plots the True Positive Rate
(Recall) against the False Positive Rate at every possible threshold:

$$\text{TPR (Recall)} = \frac{TP}{TP + FN} \qquad \text{FPR} = \frac{FP}{FP + TN}$$

The **Area Under the Curve (AUC)** summarizes the ROC in a single number:
- **AUC = 1.0:** Perfect classifier
- **AUC = 0.5:** Random guessing (the diagonal)
- **AUC < 0.5:** Worse than random (predictions are inverted)

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color='blue', linewidth=2, label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR / Recall)', fontsize=12)
ax.set_title('ROC Curve', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

> **Geometric intuition:** The ROC curve shows how much "recall" you can buy for each
> unit of "false positive rate." The curve bows toward the upper-left corner for good
> classifiers. The AUC is the probability that the model ranks a random positive
> example higher than a random negative example.

In [ ]:
# Find the threshold closest to the "optimal" point (top-left corner)
# This maximizes TPR - FPR (Youden's J statistic)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = roc_thresholds[best_idx]

print(f'Optimal threshold (Youden\'s J): {best_threshold:.3f}')
print(f'At this threshold: TPR={tpr[best_idx]:.3f}, FPR={fpr[best_idx]:.3f}')

## Precision-Recall Curve

The ROC curve can be overly optimistic when classes are **imbalanced**. A model might
look great on the ROC because TN is huge (lots of negatives to correctly reject), masking
the fact that it misses most positives.

The **Precision-Recall (PR) curve** is more informative for imbalanced datasets because
it focuses entirely on the positive class.

In [ ]:
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curve
axes[0].plot(fpr, tpr, color='blue', linewidth=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve')
axes[0].legend()

# PR curve
axes[1].plot(rec_curve, prec_curve, color='green', linewidth=2, label=f'AP = {avg_precision:.3f}')
baseline = y_test.sum() / len(y_test) if hasattr(y_test, 'sum') else sum(y_test) / len(y_test)
axes[1].axhline(baseline, color='gray', linestyle='--', alpha=0.5, label=f'Baseline = {baseline:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

> **When to use which:**
> - **ROC/AUC** is good for balanced datasets and when you care about both classes equally
> - **PR curve / Average Precision** is better for imbalanced datasets where the positive
>   class is rare and you care most about detecting it

## Threshold Tuning: Optimizing for Business Metrics

Rather than using the default 0.5 threshold, we can choose the threshold that
optimizes a **specific metric** relevant to the business problem.

In [ ]:
# Evaluate multiple metrics across all thresholds
eval_thresholds = np.arange(0.05, 0.96, 0.01)
results = []

for t in eval_thresholds:
    y_t = (y_proba >= t).astype(int)
    if y_t.sum() == 0 or y_t.sum() == len(y_t):
        continue
    results.append({
        'threshold': t,
        'accuracy': accuracy_score(y_test, y_t),
        'precision': precision_score(y_test, y_t, zero_division=0),
        'recall': recall_score(y_test, y_t),
        'f1': f1_score(y_test, y_t),
    })

df_results = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(12, 6))
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    ax.plot(df_results['threshold'], df_results[metric], linewidth=2, label=metric.title())

ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')

# Mark the F1-optimal threshold
best_f1_row = df_results.loc[df_results['f1'].idxmax()]
ax.axvline(best_f1_row['threshold'], color='red', linestyle=':', alpha=0.7,
           label=f'Best F1 threshold ({best_f1_row["threshold"]:.2f})')

ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Metrics vs Threshold')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

print(f'Best F1 threshold: {best_f1_row["threshold"]:.2f} (F1={best_f1_row["f1"]:.4f})')

## Class Imbalance

When one class vastly outnumbers the other, the model tends to be biased toward the
majority class. Several strategies address this:

### 1. `class_weight='balanced'`

Scikit-learn can automatically reweight samples inversely proportional to class
frequency. This makes the minority class "count more" in the loss function.

In [ ]:
# Create a heavily imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42,
)

X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

scaler_imb = StandardScaler()
X_imb_train = scaler_imb.fit_transform(X_imb_train)
X_imb_test = scaler_imb.transform(X_imb_test)

print(f'Class distribution: {np.bincount(y_imb)}')
print(f'Minority class: {np.bincount(y_imb)[1] / len(y_imb) * 100:.1f}%')

In [ ]:
# Compare unweighted vs balanced
model_default = LogisticRegression(random_state=42, max_iter=1000)
model_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)

model_default.fit(X_imb_train, y_imb_train)
model_balanced.fit(X_imb_train, y_imb_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, m, title in zip(
    axes,
    [model_default, model_balanced],
    ['Default (no weighting)', 'class_weight="balanced"'],
):
    y_imb_pred = m.predict(X_imb_test)
    ConfusionMatrixDisplay.from_predictions(y_imb_test, y_imb_pred, ax=ax, cmap='Blues')
    ax.set_title(title)
    recall = recall_score(y_imb_test, y_imb_pred)
    prec = precision_score(y_imb_test, y_imb_pred, zero_division=0)
    ax.set_xlabel(f'Precision={prec:.3f}, Recall={recall:.3f}')

plt.tight_layout()
plt.show()

### 2. SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE creates **synthetic** samples for the minority class by interpolating between
existing minority samples and their nearest neighbors. It is available in the
`imbalanced-learn` library:

```python
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
```

**Important:** Only apply SMOTE to the **training set**, never to the test set.

Other strategies include:
- **Random undersampling:** Remove majority class samples
- **Random oversampling:** Duplicate minority class samples
- **Tomek links / Edited Nearest Neighbors:** Clean noisy boundary samples

## Regularization in Logistic Regression

Regularization prevents overfitting by penalizing large coefficients. In scikit-learn's
`LogisticRegression`, the regularization strength is controlled by the parameter **C**,
which is the **inverse** of the regularization strength $\lambda$:

$$C = \frac{1}{\lambda}$$

- **Small C** (e.g., 0.01) = **strong** regularization = simpler model (coefficients shrunk toward zero)
- **Large C** (e.g., 100) = **weak** regularization = model fits training data more closely
- **C = infinity** = no regularization (like standard MLE)

In [ ]:
# Visualize the effect of C on coefficients
C_values = np.logspace(-3, 3, 50)
coef_paths = []

for c in C_values:
    m = LogisticRegression(C=c, penalty='l2', random_state=42, max_iter=5000, solver='lbfgs')
    m.fit(X_train, y_train)
    coef_paths.append(m.coef_[0])

coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(12, 6))
for i in range(coef_paths.shape[1]):
    ax.plot(C_values, coef_paths[:, i], linewidth=1.5, label=f'Feature {i}')

ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization strength)', fontsize=12)
ax.set_ylabel('Coefficient value', fontsize=12)
ax.set_title('Regularization Path: Coefficient Magnitude vs C', fontsize=13)
ax.axhline(0, color='black', linewidth=0.5)
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> As C increases (less regularization), the coefficients grow toward their unconstrained
> values. As C decreases (more regularization), all coefficients are shrunk toward zero.

## L1 vs L2 Penalty

Scikit-learn supports two main regularization penalties:

**L2 (Ridge)** — default: $\text{penalty} = \lambda \sum w_j^2$
- Shrinks all coefficients smoothly toward zero
- No coefficient is ever exactly zero
- Works with all solvers

**L1 (Lasso):** $\text{penalty} = \lambda \sum |w_j|$
- Can drive coefficients **exactly to zero** — automatic feature selection
- Produces **sparse** models
- Requires `solver='liblinear'` or `solver='saga'`

**Elastic Net (L1 + L2):** combines both penalties
- Controlled by `l1_ratio` parameter
- Requires `solver='saga'`

In [ ]:
# Compare L1 and L2 regularization
model_l1 = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', random_state=42, max_iter=5000)
model_l2 = LogisticRegression(penalty='l2', C=0.1, solver='lbfgs', random_state=42, max_iter=5000)

model_l1.fit(X_train, y_train)
model_l2.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feature_names = [f'Feature {i}' for i in range(X_train.shape[1])]

axes[0].barh(feature_names, model_l2.coef_[0], color='steelblue')
axes[0].set_title(f'L2 Regularization (C=0.1)\n{sum(model_l2.coef_[0] != 0)} non-zero coefficients')
axes[0].set_xlabel('Coefficient')

colors = ['steelblue' if c != 0 else 'lightgray' for c in model_l1.coef_[0]]
axes[1].barh(feature_names, model_l1.coef_[0], color=colors)
axes[1].set_title(f'L1 Regularization (C=0.1)\n{sum(model_l1.coef_[0] != 0)} non-zero coefficients')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

print('L1 coefficients (note the zeros):')
for name, coef in zip(feature_names, model_l1.coef_[0]):
    marker = ' <-- zero' if coef == 0 else ''
    print(f'  {name}: {coef:>8.4f}{marker}')

### Solver Compatibility

| Solver | L1 | L2 | Elastic Net | Multinomial |
|---|---|---|---|---|
| `lbfgs` (default) | No | Yes | No | Yes |
| `liblinear` | Yes | Yes | No | No (OvR only) |
| `saga` | Yes | Yes | Yes | Yes |
| `newton-cg` | No | Yes | No | Yes |

> **Rule of thumb:** Use `lbfgs` (the default) for L2. Switch to `liblinear` for L1
> on small datasets, or `saga` for L1/ElasticNet on large datasets.

## Hyperparameter Tuning with GridSearchCV

Instead of guessing C and the penalty, we can systematically search over a grid of
hyperparameters using cross-validation.

In [ ]:
# Define the parameter grid
param_grid = [
    {
        'penalty': ['l2'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs'],
    },
    {
        'penalty': ['l1'],
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear'],
    },
]

grid_search = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=5000),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',  # optimize for F1
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

In [ ]:
# Visualize grid search results
results_df = pd.DataFrame(grid_search.cv_results_)

fig, ax = plt.subplots(figsize=(12, 5))

for penalty in ['l1', 'l2']:
    mask = results_df['param_penalty'] == penalty
    subset = results_df[mask].sort_values('param_C')
    ax.plot(
        subset['param_C'].astype(float),
        subset['mean_test_score'],
        'o-',
        linewidth=2,
        markersize=6,
        label=f'{penalty.upper()} (test)',
    )
    ax.fill_between(
        subset['param_C'].astype(float),
        subset['mean_test_score'] - subset['std_test_score'],
        subset['mean_test_score'] + subset['std_test_score'],
        alpha=0.15,
    )

ax.set_xscale('log')
ax.set_xlabel('C (inverse regularization strength)')
ax.set_ylabel('Mean CV F1 Score')
ax.set_title('Grid Search Results: F1 Score vs C')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best model on the test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print('Best model on test set:')
print(classification_report(y_test, y_pred_best))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, ax=ax, cmap='Blues')
ax.set_title(f'Tuned Model (C={grid_search.best_params_["C"]}, {grid_search.best_params_["penalty"]})')
plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | Detail |
|---|---|
| **Confusion matrix** | Foundation for all classification metrics: TP, TN, FP, FN |
| **Precision** | Of all predicted positives, how many are correct? |
| **Recall** | Of all actual positives, how many did we find? |
| **F1 score** | Harmonic mean of precision and recall — single balanced metric |
| **ROC / AUC** | Threshold-independent measure; good for balanced datasets |
| **PR curve** | Better than ROC for imbalanced data; focuses on positive class |
| **Threshold tuning** | Choose based on business cost of FP vs FN |
| **class_weight** | Quick fix for imbalance; SMOTE for more sophisticated resampling |
| **C parameter** | Inverse of regularization strength; tune with cross-validation |
| **L1 vs L2** | L1 for sparsity/feature selection; L2 for smooth shrinkage |

---

**Next up:** Extending logistic regression to multiclass problems.